# sLTN Running Example — Weakly Supervised Video Classification

A video shows an MNIST digit being revealed frame by frame. In some videos the digit *appears* and stays; in others it appears and then disappears again. Each video comes with a single coarse label: the drawn digit for appearing videos, `unknown` otherwise.

From this video-level label alone, we want to learn two **frame-level** symbols that are never supervised directly:

- `digit(x[t])`: which digit is visible at frame `t` (or `unknown` if it is not fully drawn);
- `Complete(x[t])`: whether the digit is fully drawn at frame `t`.

The link between the coarse label and the frame-level symbols is a small knowledge base of temporal constraints. The example shows the three ingredients of sLTN:

1. a structural dimension `T` (time) declared in the signature and named in formulas (`x[t]`, `∀ t`, `next(t, t1)`, `x[t][t=0]`);
2. symbols whose declared dimensions constrain their groundings (a per-frame CNN vs. a whole-video GRU);
3. training a set of neural networks by maximising the satisfaction of the knowledge base.

Paper: **[arXiv:2608.11136](https://arxiv.org/abs/2608.11136)**.

## 1  Setup

Hyperparameters, imports and seeding.

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
NUM_SAMPLES_TRAIN = 3000
NUM_SAMPLES_VAL   = 600
NUM_SAMPLES_TEST  = 600
BATCH_SIZE        = 32
TIME_STEPS        = 24     # length of a video
COMPLETE_FRAC     = 0.85   # a digit counts as "complete" once this much of it is revealed
APPEAR_FRAC       = 0.6    # fraction of videos that are genuine "appear" (monotone reveal);
                           # the rest "appear-then-disappear" (digit fades back out)
NOISE_STD         = 0.03
RANDOM_SEED       = 2

# ── Training ───────────────────────────────────────────────────────────────
NUM_EPOCHS   = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-5
MAX_NORM     = 1.0
PATIENCE     = 8

# ── Model ──────────────────────────────────────────────────────────────────
FRAME_FEATURE_DIM = 64
RNN_HIDDEN_DIM    = 64
N_DIGITS          = 10
N_CLASSES         = N_DIGITS + 1   # classes 0-9 = real digits; class 10 = "unknown/incomplete"

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import sltn
from sltn import Signature, Interpretation, Logic, KB, kb_describe, kb_backward
import sltn.interpretation.operators as ops


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(RANDOM_SEED)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

## 2  Signature

| Symbol | Kind | Declaration | Meaning |
|---|---|---|---|
| `Image`, `Digit` | sorts | | a frame; an 11-class digit distribution (0–9 and `unknown`) |
| `T` | dimension | | time steps of a video |
| `t` | structural variable | over `T` | a time point |
| `x` | variable | `Image`, dims `T` | a video (a sequence of frames) |
| `y` | variable | `Digit` | the video-level label |
| `unknown` | constant | `Digit` | "not a digit": an incomplete frame, or a non-appearing video |
| `digit` | function | `Image → Digit` | per-frame digit classifier |
| `Complete` | predicate | `Image` | "the digit is fully drawn in this frame" |
| `appear` | predicate | `Image × Digit`, consumes `T` | "`x` is a video of digit `y` appearing" |
| `=d` | infix predicate | `Digit × Digit` | agreement between two digit distributions |
| `next` | structural relation | `T × T` | consecutive time points |

**Dimensions determine the groundings.** `digit` and `Complete` declare no dimensions, so they act frame by frame: applied to `x[t]` they return one value per frame, and their natural grounding is a per-frame CNN. `appear` instead consumes `T` (`input_dims=["T"]`, `output_dims=[]`): it receives the whole sequence and returns one value per video, so it needs a model that reads all frames, here a GRU.

In [ ]:
sig = Signature("video")

sig.sort("Image")
sig.sort("Digit")
sig.dimension("T")

sig.variable("x", "Image", dims=["T"])   # a video
sig.variable("y", "Digit")               # its label
sig.structural_variable("t", "T")
sig.constant("unknown", "Digit")

sig.function("digit", ["Image"], "Digit")                                      # per frame
sig.predicate("Complete", ["Image"])                                           # per frame
sig.predicate("appear", ["Image", "Digit"], input_dims=["T"], output_dims=[])  # whole video
sig.predicate("=d", ["Digit", "Digit"], infix=True)
sig.structural_relation("next", ["T", "T"])

print(sig)

## 3  Knowledge base

We first define a macro (a formula abbreviation, not a new symbol):

$$
\mathrm{is\_appear}(x) := \neg\,\mathrm{appear}(x, \mathrm{unknown})
$$

"according to `appear`, `x` is an appearing-digit video". The knowledge base $\mathcal K$ has five clauses:

| Clause | Formula | Role |
|---|---|---|
| $\delta$ | $\forall (x, y): \mathrm{appear}(x, y)$ | the **only supervised clause**: `appear` must match the label |
| $\phi$ | $\forall (x, y): \mathrm{appear}(x, y) \wedge \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\mathrm{Complete}(x[t]) \rightarrow \mathrm{digit}(x[t]) =_d y\bigr)$ | complete frames show the video's digit |
| $\psi$ | $\forall x: \mathrm{is\_appear}(x) \rightarrow \forall t, t_1 \mid \mathrm{next}(t, t_1): \bigl(\mathrm{Complete}(x[t]) \rightarrow \mathrm{Complete}(x[t_1])\bigr)$ | once complete, always complete |
| $\chi$ | $\forall x: \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\neg\,\mathrm{Complete}(x[t]) \rightarrow \mathrm{digit}(x[t]) =_d \mathrm{unknown}\bigr)$ | incomplete frames are `unknown` |
| $\beta$ | $\forall x: \mathrm{is\_appear}(x) \rightarrow \bigl(\neg\,\mathrm{Complete}(x[t])[t{=}0] \wedge \mathrm{Complete}(x[t])[t{=}{-1}]\bigr)$ | incomplete at the first frame, complete at the last |

**How supervision flows.** Only $\delta$ touches the label `y` as ground truth. The other clauses constrain `digit` and `Complete` relative to `appear`'s *own prediction*, so the frame-level symbols are learned indirectly. The dataset fields `label` and `complete_gt` are never used for training; we use them only to evaluate the frame-level symbols after training (§7).

**Why the `is_appear` gate.** Disappearing videos break $\psi$ and $\beta$ by construction, so these constraints are stated only for videos that `appear` judges to be appearing. The gate also matters for $\phi$: a fully visible frame looks the same in both kinds of video, and without the gate it would be pushed towards the digit in one case and towards `unknown` in the other, which no per-frame classifier can satisfy.

**Why "every complete frame" in $\phi$.** Requiring the match at *all* frames where `Complete` holds prevents a trivial solution: if `Complete` fired too early, `digit` would have to classify partially drawn frames.

In code, `psi`, `chi` and `beta` are macros of `x`, and the clauses are collected with `sltn.KB`. `x[t][t=-1]` selects the last time step.

In [ ]:
sig.define("is_appear", ["x"], "~appear(x[t], unknown)", input_dims=["T"], output_dims=[])

sig.define("psi", ["x"], "(is_appear(x[t]) -> (∀ t, t1 | next(t, t1): (Complete(x[t]) -> Complete(x[t1]))))")
sig.define("chi", ["x"], "(is_appear(x[t]) -> (∀ t: ((¬Complete(x[t]) -> (digit(x[t]) =d unknown)))))", input_dims=["T"], output_dims=[])
sig.define("beta", ["x"], "is_appear(x[t]) -> (~Complete(x[t][t=0]) & Complete(x[t][t=-1]))")

delta = sig.parse("∀ (x, y): appear(x[t], y)")
phi   = sig.parse("∀ (x, y): ((appear(x[t], y) & is_appear(x[t])) -> (∀ t: (Complete(x[t]) -> digit(x[t]) =d y)))")
psi   = sig.parse("∀ x: psi(x)")
chi   = sig.parse("∀ x: chi(x)")
beta  = sig.parse("∀ x: beta(x)")

print("delta:", delta)
print("phi:  ", phi)
print("psi:  ", psi)
print("chi:  ", chi)
print("beta: ", beta)
print()
print("fv(delta):", delta.free_variables, " fsv(delta):", delta.free_structural_variables)
print("fv(phi)  :", phi.free_variables,   " fsv(phi)  :", phi.free_structural_variables)
print("fv(psi)  :", psi.free_variables,   " fsv(psi)  :", psi.free_structural_variables)

# K = {delta, phi, ∀x:psi, ∀x:chi, ∀x:beta}
phi_kb = sltn.KB([delta, phi, psi, chi, beta])

print("\nKB:", phi_kb)
kb_dict = kb_describe(phi_kb)

## 4  Semantics

**Types.** `Image` is a $1 \times 28 \times 28$ grayscale frame. `Digit` is a vector of 11 probabilities: indices 0–9 are digits, index 10 is `unknown`. `=d` is the dot product of two such vectors, and `unknown` is the one-hot vector at index 10.

**Logic.** We use the default operators, except for implication: `ImpliesGoguen` instead of `ImpliesReichenbach`. Goguen implication is fully satisfied as soon as $a \le b$ and then stops producing gradient, whereas Reichenbach keeps pushing the antecedent down, which here means pushing `is_appear` towards false. Over five seeds, the per-frame `digit` accuracy is 0.92 ± 0.05 with Goguen and 0.52 ± 0.29 with Reichenbach. The structural quantifier `∀ t` uses the default `Forall` operator.

In [ ]:
logic = Logic().with_defaults()
logic["->"] = ops.ImpliesGoguen()

interp = Interpretation(sig, logic=logic).to(device)

interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
interp["Digit"] = sltn.Type("Digit", shape=(N_CLASSES,), axis_names=["digit_prob"])

# Dot product, clamped to [0, 1] to guard against softmax/float32 rounding slightly outside bounds.
interp["=d"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)

# Ground the "unknown" constant: one-hot at position N_DIGITS (the last class).
_unknown_vec = torch.zeros(N_CLASSES, device=device)
_unknown_vec[N_DIGITS] = 1.0
interp["unknown"] = _unknown_vec

print(interp)

## 5  Dataset

Each video has `TIME_STEPS` frames of one MNIST digit and is of one of two kinds (probabilities `APPEAR_FRAC` and `1 - APPEAR_FRAC`):

- **Appear**: blank, then the digit is revealed linearly over a random window, then it stays fully visible. Label `y` = the digit.
- **Disappear**: the same first reveal, followed by random hide/reveal cycles. Label `y` = `unknown`.

Each sample also stores `label` (the drawn digit) and `complete_gt` (whether at least `COMPLETE_FRAC` of the digit is visible at each frame), used only for evaluation. Use the sliders below to browse frames and the ground-truth `complete_gt` trace.

In [ ]:
class AppearVideoDataset(Dataset):
    """Wraps an MNIST split into two kinds of videos.

    - "Appear" (probability `appear_frac`): a single reveal window [t_start, t_end] —
      blank before t_start, linearly revealed from t_start to t_end, fully revealed
      forever after. `complete_gt` is monotone. `y = onehot(label)`.
    - "Disappear" (probability 1 - appear_frac): the same initial reveal window,
      followed by a *random number* (1-3) of further hide/reveal cycles — the digit
      fades out, may fade back in, may fade out again, etc. — at random positions,
      durations, and with a random final state (it may end the video visible or
      hidden). `complete_gt` is generally non-monotone. `y = onehot(unknown)` — this
      is not a genuine "appearing digit" video, even though a real digit was drawn
      in it and may well be visible in the final frame.

    Only `y` (and, implicitly, `is_appear`) feed any formula. `label`/`complete_gt`
    are recorded purely for post-training probing of `digit`/`Complete`.
    """

    def __init__(self, mnist_split, num_samples, time_steps, complete_frac,
                 appear_frac=1.0, noise_std=0.0, seed=0):
        self.mnist_split   = mnist_split
        self.time_steps    = time_steps
        self.complete_frac = complete_frac
        self.appear_frac   = appear_frac
        self.noise_std     = noise_std
        self.seed          = seed

        rng = random.Random(seed)
        n_available = len(mnist_split)
        self.indices    = [rng.randrange(n_available) for _ in range(num_samples)]
        self.is_appear  = [rng.random() < appear_frac for _ in range(num_samples)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.mnist_split[self.indices[idx]]
        H = image.shape[-2]
        is_appear = self.is_appear[idx]
        T = self.time_steps
        quarter = max(1, T // 4)

        rng_local = random.Random(self.seed * 1_000_003 + idx)
        t_start = rng_local.randrange(0, quarter + 1)
        t_end   = min(t_start + rng_local.randrange(0, quarter + 1), T // 2 - 1)

        reveal_frac = torch.zeros(T)
        if t_end > t_start:
            reveal_frac[t_start:t_end + 1] = torch.linspace(0.0, 1.0, steps=t_end - t_start + 1)
        else:
            reveal_frac[t_start] = 1.0
        reveal_frac[t_end + 1:] = 1.0    # fully revealed after t_end

        if not is_appear:
            # A random number of further hide/reveal cycles after the initial reveal:
            # fade out, hold, maybe fade back in, hold, fade out again, … — random
            # count, positions, and durations, ending in an arbitrary (visible or
            # hidden) final state. None of this is ever supervised directly; it just
            # produces varied, generally non-monotone `complete_gt` traces that serve
            # as counterexamples to psi (which only applies to genuine `is_appear`
            # videos).
            n_cycles = rng_local.randrange(1, 4)   # 1..3 fade transitions
            cursor   = t_end + 1
            visible  = True                        # state coming out of the initial reveal
            for _ in range(n_cycles):
                remaining = T - 1 - cursor
                if remaining <= 0:
                    break
                start_val, end_val = (1.0, 0.0) if visible else (0.0, 1.0)

                ramp_dur = rng_local.randrange(1, max(2, remaining // 2 + 1))
                ramp_end = min(cursor + ramp_dur, T - 1)
                if ramp_end > cursor:
                    reveal_frac[cursor:ramp_end + 1] = torch.linspace(start_val, end_val, steps=ramp_end - cursor + 1)
                else:
                    reveal_frac[cursor] = end_val

                hold_remaining = T - 1 - ramp_end
                hold_dur = rng_local.randrange(0, hold_remaining // 2 + 1) if hold_remaining > 0 else 0
                hold_end = min(ramp_end + hold_dur, T - 1)
                reveal_frac[ramp_end + 1:hold_end + 1] = end_val

                cursor, visible = hold_end + 1, not visible
                if cursor >= T - 1:
                    break

        frames = torch.zeros(T, *image.shape)
        for t, frac in enumerate(reveal_frac):
            n_rows = int(round(frac.item() * H))
            frames[t, :, :n_rows, :] = image[:, :n_rows, :]
        if self.noise_std > 0:
            frames = (frames + self.noise_std * torch.randn_like(frames)).clamp(0.0, 1.0)

        # y is a SINGLE per-video one-hot over N_CLASSES (no T!) — digit-index if this is
        # a genuine appearing-digit video, unknown-index otherwise. This is the only
        # ground truth any formula ever sees (via delta/phi's appear(x, y)).
        y_onehot = torch.zeros(N_CLASSES)
        y_onehot[label if is_appear else N_DIGITS] = 1.0

        complete_gt = (reveal_frac >= self.complete_frac).float().unsqueeze(-1)

        return {
            "x": frames, "y": y_onehot, "complete_gt": complete_gt,
            "label": label, "is_appear": float(is_appear),
        }


frame_size = (28, 28)
mnist_train_full = datasets.MNIST("../datasets/", train=True, download=True,
    transform=transforms.Compose([transforms.Resize(frame_size), transforms.ToTensor()]),
)
train_split_raw, val_split_raw = random_split(
    mnist_train_full, [50000, 10000],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
mnist_test = datasets.MNIST("../datasets/", train=False, download=True,
    transform=transforms.Compose([transforms.Resize(frame_size), transforms.ToTensor()]),
)

train_ds = AppearVideoDataset(train_split_raw, NUM_SAMPLES_TRAIN, TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED)
val_ds   = AppearVideoDataset(val_split_raw,   NUM_SAMPLES_VAL,   TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED + 1)
test_ds  = AppearVideoDataset(mnist_test,       NUM_SAMPLES_TEST,  TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED + 2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

n_appear_train = sum(train_ds.is_appear)
print(f"Train: {len(train_ds)} ({n_appear_train} appear, {len(train_ds) - n_appear_train} disappear)")
print(f"Val:   {len(val_ds)}  |  Test: {len(test_ds)}")
print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

In [ ]:
from ipywidgets import interact, IntSlider, fixed

def plot_sample_gt(sample_idx, time_step, dataset):
    sample = dataset[sample_idx]
    frames = sample["x"]                       # (T, 1, 28, 28)
    time_np = np.arange(TIME_STEPS)
    complete_np = sample["complete_gt"].squeeze(-1).numpy()

    fig, (ax_img, ax_tr) = plt.subplots(1, 2, figsize=(10, 4))

    kind = "appear" if sample["is_appear"] else "disappear"
    ax_img.imshow(frames[time_step, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax_img.set_title(f"frame t={time_step}  (label={sample['label']}, type={kind})")
    ax_img.axis("off")

    ax_tr.plot(time_np, complete_np, marker="o", color="tab:green")
    ax_tr.axvline(time_step, color="red", ls="--", alpha=0.7, label="current t")
    ax_tr.set_title(f"Ground-truth Complete(x)[t]  ({kind})")
    ax_tr.set_xlabel("time"); ax_tr.set_ylabel("truth value")
    ax_tr.set_ylim(-0.05, 1.1); ax_tr.legend(); ax_tr.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()


interact(
    plot_sample_gt,
    sample_idx=IntSlider(min=0, max=len(train_ds) - 1, step=1, value=0, description="Sample"),
    time_step=IntSlider(min=0, max=TIME_STEPS - 1, step=1, value=TIME_STEPS // 2, description="Time step"),
    dataset=fixed(train_ds),
)
plt.show()

## 6  Models and training

As required by the signature (§2), `digit` and `Complete` are per-frame CNNs and receive frames with batch and time flattened, `(B·T, 1, 28, 28)`. `appear` is a CNN encoder followed by a GRU; it receives whole videos `(B, T, 1, 28, 28)` and outputs a distribution over the 11 classes, and `appear(x, d)` is its dot product with `d`.

In [ ]:
class FrameEncoder(nn.Module):
    """Small CNN applied independently to each frame of a (B, T, 1, H, W) video."""

    def __init__(self, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 28 -> 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 14 -> 7
        )
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, out_dim), nn.ReLU())

    def forward(self, x):
        # `digit`/`Complete` declare no free `T` dependence, so the interpreter has
        # already flattened `T` into the batch dim before calling us: x is 4D,
        # (N, 1, H, W). `appear`'s GRU instead needs the *whole* video, so the
        # interpreter passes `x` through untouched: x is 5D, (B, T, 1, H, W). Handle
        # both by flattening/restoring `T` ourselves around the frame-local CNN.
        if x.dim() == 5:
            B, T = x.shape[:2]
            feats = self.proj(self.conv(x.reshape(B * T, *x.shape[2:])))
            return feats.reshape(B, T, -1)
        return self.proj(self.conv(x))


class DigitClassifier(nn.Module):
    """Grounds the `digit` function: frame(s) (B, T, 1, H, W) -> per-frame digit distribution
    (B, T, N_CLASSES). Purely frame-local — no GRU, no dependency across T whatsoever,
    matching `digit`'s signature (no structural dimensions declared at all).

    Outputs N_CLASSES = N_DIGITS + 1 probabilities per frame; the last class (index
    N_DIGITS) is the "unknown" pseudo-class.
    """

    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.head = nn.Linear(feature_dim, n_classes)

    def forward(self, x):
        feats = self.encoder(x)                        # (B, feature_dim)
        return torch.softmax(self.head(feats), dim=-1)  # (B, n_classes)


class CompleteClassifier(nn.Module):
    """Grounds the `Complete` predicate: frame(s) (B, T, 1, H, W) -> truth degree (B, T, 1)."""

    def __init__(self, feature_dim):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.head = nn.Sequential(nn.Linear(feature_dim, 1), nn.Sigmoid())

    def forward(self, x):
        feats = self.encoder(x)          # (..., T, feature_dim)
        return self.head(feats)          # (..., T, 1)


class AppearClassifier(nn.Module):
    """Grounds the whole-video part of `appear`: video (B, T, 1, H, W) -> a single
    softmax distribution (B, N_CLASSES) over "which digit is genuinely appearing, or
    none". A GRU is run across the *entire* time axis and only its final hidden state is
    read out — the model structurally cannot answer without having seen every frame.
    """

    def __init__(self, feature_dim, hidden_dim, n_classes):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.rnn = nn.GRU(feature_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        feats = self.encoder(x)            # (B, T, feature_dim)
        _, h_n = self.rnn(feats)           # h_n: (1, B, hidden_dim)
        return torch.softmax(self.head(h_n[-1]), dim=-1)   # (B, n_classes)


digit_model    = DigitClassifier(FRAME_FEATURE_DIM, N_CLASSES).to(device)
complete_model = CompleteClassifier(FRAME_FEATURE_DIM).to(device)
appear_model   = AppearClassifier(FRAME_FEATURE_DIM, RNN_HIDDEN_DIM, N_CLASSES).to(device)

interp["digit"]    = digit_model
interp["Complete"] = complete_model


class AppearPredicate(nn.Module):
    """appear(x, d): dot product between the whole-video softmax and the (typically
    one-hot) digit vector d — the same trick already used for `=d`.

    `appear` consumes `T`, and every argument receives the consumed axes of the call,
    so `d` arrives as `(B, T, N_CLASSES)` with a copy at every step. `appear_model`
    removes `T` before the two are combined, so the copies are dropped first.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, d):
        probs = self.model(x)
        d = d.select(1, 0)
        return (probs * d).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)


interp["appear"] = AppearPredicate(appear_model)

# Structural relation `next`: adjacency mask over consecutive time indices.
next_mask = torch.zeros(TIME_STEPS, TIME_STEPS, device=device)
for i in range(TIME_STEPS - 1):
    next_mask[i, i + 1] = 1.0
interp["next"] = next_mask

optimizer = torch.optim.Adam(interp.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(interp)

Each training step binds a batch to `x` and `y` and calls `kb_backward`, which minimises `1 - SatAgg` over the five clauses. The reported accuracy is that of `appear` (argmax of its output vs. `y`), the only directly supervised quantity.

In [ ]:
@torch.no_grad()
def classify_appear(interp_):
    """Predict argmax_d appear_model(x)[d] — the only quantity delta directly supervises.
    Index N_DIGITS ("unknown") means "not a genuine appearing-digit video"."""
    return appear_model(interp_["x"]).argmax(dim=-1)   # (B,)


@torch.no_grad()
def classify_digit_at_complete(interp_, complete_gt):
    """Probe `digit`: argmax over real digit classes, at each frame, then majority-vote
    across frames where complete_gt == 1 (falls back to the globally most confident frame
    if a video has no ground-truth-complete frame at all). NEVER used in training."""
    probs = interp_["digit"](interp_["x"])                       # (B, T, N_CLASSES)
    digit_probs = probs[..., :N_DIGITS]                          # (B, T, N_DIGITS)
    per_frame_pred = digit_probs.argmax(dim=-1)                   # (B, T)
    mask = complete_gt.squeeze(-1) > 0.5                          # (B, T)
    preds = []
    for b in range(per_frame_pred.shape[0]):
        idxs = mask[b].nonzero(as_tuple=True)[0]
        if len(idxs) > 0:
            vals, counts = per_frame_pred[b, idxs].unique(return_counts=True)
            preds.append(vals[counts.argmax()].item())
        else:
            preds.append(digit_probs[b].max(dim=-1).values.argmax().item())
    return torch.tensor(preds, device=per_frame_pred.device)


# ── Smoke-test: one forward pass through the full 5-clause KB ─────────────
_batch = next(iter(train_loader))
interp["x"] = _batch["x"].to(device)
interp["y"] = _batch["y"].to(device)

with torch.no_grad():
    sat = interp(phi_kb).value
print(f"KB satisfaction (untrained): {sat.mean().item():.4f}  (expect ~0.5)")
del _batch

In [ ]:
best_val_loss  = float("inf")
patience_count = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    digit_model.train(); complete_model.train(); appear_model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    for batch in tqdm(train_loader, desc=f"Ep {epoch+1:3d} [train]", leave=False):
        optimizer.zero_grad()

        interp["x"] = batch["x"].to(device)
        interp["y"] = batch["y"].to(device)

        stats = kb_backward(optimizer, interp, kb_dict, aggregator = "satagg")
        torch.nn.utils.clip_grad_norm_(
            list(digit_model.parameters()) + list(complete_model.parameters())
            + list(appear_model.parameters()),
            MAX_NORM,
        )
        with torch.no_grad():
            preds = classify_appear(interp)
        optimizer.step()

        train_loss += stats["agg_loss"]
        train_preds.extend(preds.cpu().tolist())
        train_labels.extend(batch["y"].argmax(dim=-1).tolist())   # appear ground truth

    train_loss /= len(train_loader)
    train_acc = sum(p == l for p, l in zip(train_preds, train_labels)) / len(train_labels)

    # ── Validate ───────────────────────────────────────────────────────────
    digit_model.eval(); complete_model.eval(); appear_model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            interp["x"] = batch["x"].to(device)
            interp["y"] = batch["y"].to(device)

            sat = interp(phi_kb).value
            val_loss += (1.0 - sat).item()
            val_preds.extend(classify_appear(interp).cpu().tolist())
            val_labels.extend(batch["y"].argmax(dim=-1).tolist())

    val_loss /= len(val_loader)
    val_acc = sum(p == l for p, l in zip(val_preds, val_labels)) / len(val_labels)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    tqdm.write(
        f"Ep {epoch+1:3d} | train loss {train_loss:.4f} "
        f"| train acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f} | val acc {val_acc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss, patience_count = val_loss, 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}")
            break

## 7  Evaluation

We report the accuracy of `appear`, then evaluate the two frame-level symbols against the held-out fields: `digit` on the complete frames of appearing videos (vs. `label`), and `Complete` (vs. `complete_gt`).

In [ ]:
# ── Learning curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("Loss (1 - SatAgg)"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["val_acc"], color="tab:green")
axes[1].set_title("Val accuracy"); axes[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# ── (1) `appear` classification report — the ONLY directly-supervised quantity ─────
digit_model.eval(); complete_model.eval(); appear_model.eval()

appear_preds, appear_labels = [], []
digit_preds, digit_labels, digit_is_appear = [], [], []
complete_preds, complete_gts, complete_is_appear = [], [], []

with torch.no_grad():
    for batch in test_loader:
        interp["x"] = batch["x"].to(device)
        complete_gt = batch["complete_gt"].to(device)

        appear_preds.extend(classify_appear(interp).cpu().tolist())
        appear_labels.extend(batch["y"].argmax(dim=-1).tolist())

        digit_preds.extend(classify_digit_at_complete(interp, complete_gt).cpu().tolist())
        digit_labels.extend(batch["label"].tolist())
        digit_is_appear.extend(batch["is_appear"].tolist())

        complete_preds.append(complete_model(interp["x"])[..., 0].cpu())
        complete_gts.append(complete_gt[..., 0].cpu())
        complete_is_appear.extend(batch["is_appear"].tolist())

class_names = [str(d) for d in range(N_DIGITS)] + ["unknown"]
print("=== appear(x, y) classification — directly supervised (delta) ===")
print(classification_report(appear_labels, appear_preds, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(appear_labels, appear_preds, display_labels=class_names, ax=ax)
ax.set_title("Test confusion matrix — appear(x, y)")
plt.tight_layout(); plt.show()

# ── (2) `digit` probing — NEVER directly supervised, learned only via phi/chi ──────
digit_is_appear = torch.tensor(digit_is_appear).bool()
digit_preds_t   = torch.tensor(digit_preds)
digit_labels_t  = torch.tensor(digit_labels)


def _acc(mask):
    if mask.sum() == 0:
        return float("nan")
    return (digit_preds_t[mask] == digit_labels_t[mask]).float().mean().item()


print("\n=== digit(x[t]) probing accuracy (majority vote at complete_gt==1 frames) ===")
print(f"  overall           : {_acc(torch.ones_like(digit_is_appear)):.3f}")
print(f"  appear videos     : {_acc(digit_is_appear):.3f}")
print(f"  disappear videos  : {_acc(~digit_is_appear):.3f}  (never gated by chi/beta — pure generalisation)")

# ── (3) `Complete` probing — NEVER directly supervised, learned only via psi/beta ──
complete_preds_t = torch.cat(complete_preds, dim=0)          # (N, T)
complete_gts_t   = torch.cat(complete_gts, dim=0)             # (N, T)
complete_is_appear_t = torch.tensor(complete_is_appear).bool()
complete_err = (complete_preds_t - complete_gts_t).abs().mean(dim=-1)   # (N,) mean abs error over t

print("\n=== Complete(x)[t] probing — mean |pred - gt| over t ===")
print(f"  appear videos     : {complete_err[complete_is_appear_t].mean().item():.3f}")
print(f"  disappear videos  : {complete_err[~complete_is_appear_t].mean().item():.3f}  (psi/beta never asserted here)")

### Clause satisfaction on the test set

The satisfaction of each clause of $\mathcal K$ (§3), averaged over the test batches.

In [ ]:
delta_sats, phi_sats, psi_sats, chi_sats, beta_sats = [], [], [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        interp["x"] = batch["x"].to(device)
        interp["y"] = batch["y"].to(device)

        delta_sats.append(interp(delta).value.item())
        phi_sats.append(interp(phi).value.item())
        psi_sats.append(interp(psi).value.item())
        chi_sats.append(interp(chi).value.item())
        beta_sats.append(interp(beta).value.item())

print(f"Avg delta satisfaction           : {sum(delta_sats)/len(delta_sats):.4f}  (appear(x, y) — directly supervised)")
print(f"Avg phi   satisfaction           : {sum(phi_sats)/len(phi_sats):.4f}  (appear(x,y) & is_appear(x) -> ∀t: Complete -> digit match)")
print(f"Avg (∀x: psi)  satisfaction      : {sum(psi_sats)/len(psi_sats):.4f}  (is_appear(x) -> Complete persistence)")
print(f"Avg (∀x: chi)  satisfaction      : {sum(chi_sats)/len(chi_sats):.4f}  (is_appear(x) -> abstain when incomplete)")
print(f"Avg (∀x: beta) satisfaction      : {sum(beta_sats)/len(beta_sats):.4f}  (is_appear(x) -> boundary condition)")

### Predictions on a test video

Four panels: the most revealed frame with the true and predicted digits; `digit(x[t])` over time (dashed grey: `unknown`); `Complete(x[t])` (green) against `complete_gt` (black dashed); and `appear`'s output distribution with the true label highlighted.

In [ ]:
@torch.no_grad()
def plot_test_prediction(sample_idx, dataset):
    digit_model.eval(); complete_model.eval(); appear_model.eval()

    sample = dataset[sample_idx]
    x = sample["x"].unsqueeze(0).to(device)          # (1, T, 1, 28, 28)
    time_np = np.arange(TIME_STEPS)
    colors = plt.cm.tab10(np.linspace(0, 1, N_DIGITS))
    kind = "appear" if sample["is_appear"] else "disappear"

    all_probs     = digit_model(x)[0].cpu().numpy()           # (T, N_CLASSES)
    digit_probs   = all_probs[:, :N_DIGITS]                   # (T, N_DIGITS)
    unknown_probs = all_probs[:, N_DIGITS]                    # (T,)
    complete_pred = complete_model(x)[0, :, 0].cpu().numpy()  # (T,)
    complete_gt   = sample["complete_gt"].squeeze(-1).numpy()
    appear_probs  = appear_model(x)[0].cpu().numpy()           # (N_CLASSES,)
    pred_class    = int(digit_probs.max(axis=0).argmax())
    peak_t        = int(complete_gt.argmax()) if complete_gt.max() > 0 else int(np.argmax(digit_probs.max(axis=-1)))

    fig, (ax_img, ax_dig, ax_comp, ax_app) = plt.subplots(1, 4, figsize=(20, 4))

    ax_img.imshow(sample["x"][peak_t, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax_img.set_title(f"peak frame t={peak_t}  (true={sample['label']}, type={kind})\ndigit pred={pred_class}")
    ax_img.axis("off")

    for c in range(N_DIGITS):
        lw = 2.5 if c == sample["label"] else 1
        ax_dig.plot(time_np, digit_probs[:, c], color=colors[c], lw=lw,
                    label=f"{c}" + (" ← true" if c == sample["label"] else ""))
    ax_dig.plot(time_np, unknown_probs, color="gray", lw=1.5, ls="--", label="unknown")
    ax_dig.set_title("digit(x)[t] — never directly supervised")
    ax_dig.set_xlabel("time"); ax_dig.set_ylabel("probability")
    ax_dig.legend(ncols=2, fontsize=7); ax_dig.grid(True, alpha=0.3)

    ax_comp.plot(time_np, complete_gt,   color="black", ls="--", lw=2, label="GT Complete")
    ax_comp.plot(time_np, complete_pred, color="tab:green", lw=2, label="Pred Complete")
    ax_comp.set_title("Complete(x)[t] — never directly supervised"); ax_comp.set_xlabel("time")
    ax_comp.set_ylim(-0.05, 1.1); ax_comp.legend(); ax_comp.grid(True, alpha=0.3)

    bar_colors = ["tab:blue"] * N_DIGITS + ["gray"]
    true_appear_idx = sample["label"] if sample["is_appear"] else N_DIGITS
    bar_colors[true_appear_idx] = "tab:red"
    ax_app.bar(range(N_CLASSES), appear_probs, color=bar_colors)
    ax_app.set_xticks(range(N_CLASSES))
    ax_app.set_xticklabels([str(d) for d in range(N_DIGITS)] + ["unk"])
    ax_app.set_title("appear(x, ·) — directly supervised (δ)")
    ax_app.set_ylim(0, 1.05); ax_app.grid(True, alpha=0.3)

    fig.suptitle(f"Test sample {sample_idx}  (type={kind})")
    plt.tight_layout(); plt.show()


interact(
    plot_test_prediction,
    sample_idx=IntSlider(min=0, max=len(test_ds) - 1, step=1, value=0, description="Test sample"),
    dataset=fixed(test_ds),
)
plt.show()